## Notebook 16 — Ceiling Test di Model INSTRUCT (review A6)

Klaim paper "output nyaris tak berubah saat seluruh identitas diganti"
diukur di base model + letter-readout. Reviewer: itu bisa properti
INTERFACE, bukan properti model. Cek: ukur mulut di **Mistral-7B base vs
Mistral-7B-Instruct-v0.2** — sel, soal, dan pilihan jawaban SAMA PERSIS;
yang beda cuma model + format prompt yang wajar untuk masing-masing
(base = kelanjutan teks; instruct = chat template).

Dari prediksi per-(sel, soal) ini, analisis lokal menghitung:
- **ceiling identity-swap** per tipe: rata-rata `wd(predA, realB) − wd(predB, realB)`
  atas pasangan sel setipe di soal bersama;
- arah (D2): apakah predA lebih dekat ke realA daripada ke realB;
- akurasi absolut mulut per tipe.

Kalau ceiling tetap kecil di instruct → klaim menguat banyak. Kalau
membesar → scope klaim dipersempit eksplisit ke base+letter-readout.

Setting: GPU T4 x2, attach dataset CSV, internet ON. ~40-60 menit total.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm


In [ ]:
import os, gc, glob, ast, shutil
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = [
    ("mistral_base", "mistralai/Mistral-7B-v0.1", "raw"),
    ("mistral_instruct", "mistralai/Mistral-7B-Instruct-v0.2", "chat"),
]

_c = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
DATA_PATH = _c[0] if _c else "opinionqa_intersectional.csv"
print("Data:", DATA_PATH)

RANDOM_SEED = 42
MAX_OPTIONS = 6
MIN_COVERAGE_FRAC = 0.6
MIN_COVERAGE_ABS = 10
N_Q = 40
BATCH = 16
OUT_DIR = "/kaggle/working/ceiling_instruct"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)
GROUP_KEYS = sorted(df["group_key"].unique().tolist())
group_meta = {gk: dict(zip(("attr_type", "v1", "v2"),
                           [gk.split(" :: ")[0]] + gk.split(" :: ", 1)[1].split(" | ", 1)))
              for gk in GROUP_KEYS}
ALL_TYPES = sorted({m["attr_type"] for m in group_meta.values()})
qmeta = {r.qkey: (r.question, r.options[: r.n_opt], r.ordinal) for r in df.itertuples()}

ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}
LETTERS = ["A", "B", "C", "D", "E", "F"]

# SOAL SAMA dgn notebook 15 (seed & aturan identik) -> hasil bisa disandingkan
rng_q = np.random.default_rng(RANDOM_SEED)
plan = {}
sub_ok = df[df["n_opt"] <= MAX_OPTIONS]
for ty in ALL_TYPES:
    sub = sub_ok[sub_ok["attribute"] == ty]
    cells_ = sorted(sub["group_key"].unique().tolist())
    qpc = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    cnt = {}
    for gk in cells_:
        for qk in qpc.get(gk, set()):
            cnt[qk] = cnt.get(qk, 0) + 1
    min_cov = min(max(MIN_COVERAGE_ABS, int(np.ceil(MIN_COVERAGE_FRAC * len(cells_)))), len(cells_))
    good = sorted([q for q, c in cnt.items() if c >= min_cov])
    idx = rng_q.choice(len(good), size=min(N_Q, len(good)), replace=False)
    picked = sorted(good[i] for i in idx)
    plan[ty] = [(gk, qk) for qk in picked for gk in cells_ if qk in qpc.get(gk, set())]
    print(f"[{ty}] {len(plan[ty])} baris")
print("total per model:", sum(len(v) for v in plan.values()))

def identity_sentence(gk):
    m = group_meta[gk]
    l1, l2 = ATTR_LABELS[m["attr_type"]]
    return f"This survey respondent's {l1} is {m['v1']} and their {l2} is {m['v2']}."

def question_block(qk):
    question, options, _ = qmeta[qk]
    lines = [f"Question: {question}"]
    for i, opt in enumerate(options):
        lines.append(f"{LETTERS[i]}) {opt}")
    return "\n".join(lines)

def build_prompt(gk, qk, mode, tokenizer):
    if mode == "raw":
        return identity_sentence(gk) + "\n\n" + question_block(qk) + "\nAnswer:"
    # chat: persona jadi konteks, minta jawab satu huruf
    m = group_meta[gk]
    l1, l2 = ATTR_LABELS[m["attr_type"]]
    user = (f"You are answering a survey as a respondent whose {l1} is {m['v1']} "
            f"and whose {l2} is {m['v2']}. Answer the question with a single "
            f"letter only.\n\n" + question_block(qk))
    return tokenizer.apply_chat_template([{"role": "user", "content": user}],
                                         tokenize=False, add_generation_prompt=True)


In [ ]:
def letter_ids_for(tokenizer, variant):
    """variant 'space' -> token ' A'; 'bare' -> token 'A'."""
    ids = []
    for L in LETTERS:
        s = (" " + L) if variant == "space" else L
        ids.append(tokenizer.encode(s, add_special_tokens=False)[-1])
    return ids if len(set(ids)) == len(ids) else None

def run_model(tag, path, mode):
    out_csv = os.path.join(OUT_DIR, f"mouth_preds_{tag}.csv")
    if os.path.exists(out_csv):
        print(f"[skip] {tag}")
        return
    tokenizer = AutoTokenizer.from_pretrained(path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True)
    model.eval()

    ids_sp = letter_ids_for(tokenizer, "space")
    ids_bare = letter_ids_for(tokenizer, "bare")

    @torch.no_grad()
    def logits_last(prompts):
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        return model(**inputs).logits[:, -1, :].float()

    # kalibrasi: pilih varian id huruf dgn massa prob lebih besar (32 baris pertama)
    ty0 = ALL_TYPES[0]
    calib = [build_prompt(gk, qk, mode, tokenizer) for gk, qk in plan[ty0][:32]]
    lg = logits_last(calib)
    mass = {}
    for name, ids in [("space", ids_sp), ("bare", ids_bare)]:
        if ids is None:
            continue
        mass[name] = torch.softmax(lg, -1)[:, ids].sum(1).mean().item()
    variant = max(mass, key=mass.get)
    LETTER_IDS = ids_sp if variant == "space" else ids_bare
    print(f"[{tag}] varian huruf: {variant} (massa {mass})")

    rows = []
    for ty in ALL_TYPES:
        items = plan[ty]
        for s in tqdm(range(0, len(items), BATCH), desc=f"{tag} {ty}"):
            chunk = items[s:s + BATCH]
            # kelompokkan per n_opt biar softmax konsisten
            lg = logits_last([build_prompt(gk, qk, mode, tokenizer) for gk, qk in chunk])
            for b, (gk, qk) in enumerate(chunk):
                n_opt = len(qmeta[qk][2])
                sel = lg[b, LETTER_IDS[:n_opt]]
                p = torch.softmax(sel, dim=0).cpu().numpy()
                rows.append(dict(model=tag, ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                 pred=",".join(f"{x:.6f}" for x in p)))
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[simpan] {out_csv} ({len(rows)} baris)")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

for tag, path, mode in MODELS:
    run_model(tag, path, mode)
print("SELESAI:", sorted(os.listdir(OUT_DIR)))


## Download & analisis

Download `ceiling_instruct/mouth_preds_*.csv` →
`notebooks/output/16_ceiling_instruct_kaggle/`, lalu:

```
./venv/Scripts/python.exe analisis_lokal/ceiling_instruct.py
```

yang menghitung per model x tipe: ceiling identity-swap, uji arah (D2),
akurasi absolut — tabel perbandingan base vs instruct untuk paper §6.1
(scope klaim) dan Limitations.

Catatan: Mistral-7B-Instruct-v0.2 nggak butuh HF token (ungated); kalau
ganti ke model gated, tambahkan login dulu.
